In [ ]:
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import os
import re
import logging
from tqdm import tqdm

In [ ]:
# new subset gen    

In [ ]:
# embeddings subset generation << 

In [ ]:
# open metadata
# open dir with patient npz files
# randomly select n patients for subset
# extract all rows where metadata patient id matches the patient subset

In [ ]:
# updated embedding path 10-29-25
# full embeddings path
embedding_path = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/ckd_embedding_full_v3_icd_stage_filter/"

In [ ]:
import random
def select_random_patients(directory_path, n):
    # 1. Check if the provided path is a valid directory

    # 2. Get a list of all items in the directory
    all_items = os.listdir(directory_path)

    folder_names = all_items
    folder_names = [
        item for item in all_items 
        if not item.lower().endswith(".csv")
    ]
    
    if n >= len(folder_names):
        print(f"Warning: n ({n}) is greater than or equal to the total number of folders ({len(folder_names)}). Returning all folders found.")
        return folder_names

    # 5. Randomly select 'n' folder names
    # random.sample is efficient and ensures all selections are unique
    selected_folders = random.sample(folder_names, n)

    return selected_folders


In [ ]:
npatients = 4
patient_subset = select_random_patients(embedding_path, npatients)


In [ ]:
patient_subset

In [ ]:
metadata_file = embedding_path + "meta_v3.csv" #sep='$' # meta_v3_all.csv, meta_v3.csv
metadata = pl.read_csv(metadata_file, separator="$")

In [ ]:
metadata.head()

In [ ]:
meta_subset = metadata.filter(
        pl.col("PatientID").is_in(patient_subset)
    )

In [ ]:
meta_subset

In [ ]:
output_dir = f"./embeddings_subset_{npatients}/"
print(output_dir)
npatients

In [ ]:
try:
    os.mkdir(output_dir)
except FileExistsError:
    pass

In [ ]:
meta_subset.write_csv(os.path.join(output_dir, f"meta_v3_subset_{npatients}.csv"),  separator="$")

In [ ]:
pl.read_csv(output_dir + f"meta_v3_subset_{npatients}.csv",  separator="$")

In [ ]:
# copy randomly selected patients to subset

In [ ]:
import os
import shutil
from typing import List

def copy_selected_folders(source_dir: str, dest_dir: str, selected_folders: List[str]):
    print(f"Attempting to copy {len(selected_folders)} folders...")
    
    # Iterate through the list of folders to copy
    for folder_name in selected_folders:
        source_path = os.path.join(source_dir, folder_name)
        dest_path = os.path.join(dest_dir, folder_name)
            
        # shutil.copytree copies the folder and all its contents recursively
        shutil.copytree(source_path, dest_path)
        print(f"Copied: {folder_name}")

    print("\nCopy operation complete.")

In [ ]:
copy_selected_folders(embedding_path, output_dir, patient_subset)

In [ ]:
os.listdir(output_dir)


In [ ]:
print(output_dir)

In [ ]:
# tabular subset generation <<

In [ ]:
# full raw data path
event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"

In [ ]:
df = pl.read_csv(
    event_file,
    separator='$',
    infer_schema_length=None,
    null_values="null",
).unique()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
sample_size = 1000

patient_subset_list = df.select(pl.col('PatientID').unique()).sample(sample_size).to_series()
subset = df.filter(pl.col('PatientID').is_in(patient_subset_list))

In [ ]:
subset.shape

In [ ]:
subset.head()

In [ ]:
len(subset['PatientID'].unique())

In [ ]:
output_dir = f"./tabular_subset_{str(sample_size)}"
print(output_dir)
os.makedirs(output_dir, exist_ok=True) 

In [ ]:
new_fn = os.path.join(output_dir, f"unprocessed_tab_subset_{str(sample_size)}.csv")
print(new_fn)

In [ ]:

subset.write_csv(new_fn, separator="$")

In [ ]:
check_subset = pl.read_csv(
    new_fn,
    separator='$',
    infer_schema_length=None,
    null_values="null",
).unique()

In [ ]:
subset

In [ ]:
subset.shape

In [ ]:
len(subset['PatientID'].unique())

In [ ]:
# scrap code
# df = df.with_columns(
#     pl.col("PatientID").cast(pl.Utf8, strict=False),
#     pl.col("EventTimeStamp").cast(pl.Utf8, strict=False),
#     pl.col("DataCategory").cast(pl.Utf8, strict=False),
#     pl.col("DataNumeric").cast(data_numeric_dtype, strict=False),
#     pl.col("DataType").cast(pl.Utf8, strict=False),
# )